# Customer Churn Prediction using Random Forest
**Course:** B.Tech – Gen AI (2nd Semester)  
**Dataset:** Customertravel.csv

## 1. Introduction

**Customer churn** refers to the phenomenon where customers stop doing business with a company. In the travel and airline industry, churn can significantly impact revenue and brand loyalty.

Predicting churn in advance allows businesses to:
- Take proactive retention measures
- Allocate marketing budgets more efficiently
- Improve customer satisfaction and lifetime value

**Why Random Forest?**  
Random Forest is an ensemble learning algorithm that builds multiple decision trees and aggregates their predictions. It is ideal for this task because:
- It handles both numerical and categorical features well
- It is robust to overfitting due to bagging
- It provides feature importance scores, which aid in interpretability
- It performs well even with imbalanced datasets

## 2. Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve, auc
)

print('All libraries imported successfully!')

## 3. Data Loading and Exploration

In [ ]:
# Load dataset
df = pd.read_csv('Customertravel.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

In [ ]:
# Summary statistics
df.describe(include='all')

In [ ]:
# Data types and missing values
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())
print('\nTarget Distribution:')
print(df['Target'].value_counts())
print(f'Churn Rate: {df["Target"].mean()*100:.2f}%')

In [ ]:
# Unique values in categorical columns
cat_cols = ['FrequentFlyer', 'AnnualIncomeClass', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']
for col in cat_cols:
    print(f'{col}: {df[col].unique().tolist()}')

## 4. Data Cleaning and Preprocessing

In [ ]:
# No missing values found - confirm
assert df.isnull().sum().sum() == 0, 'Missing values detected!'
print('No missing values - dataset is clean.')

# Create a copy for preprocessing
df_processed = df.copy()

# Encode categorical features using Label Encoding
le = LabelEncoder()
encoders = {}

for col in cat_cols:
    encoders[col] = LabelEncoder()
    df_processed[col] = encoders[col].fit_transform(df_processed[col])
    print(f'{col} encoded: {dict(zip(encoders[col].classes_, encoders[col].transform(encoders[col].classes_)))}')

print('\nTarget column already binary (0 = No Churn, 1 = Churn)')

In [ ]:
# Define features and target
X = df_processed.drop('Target', axis=1)
y = df_processed['Target']

print(f'Feature columns: {X.columns.tolist()}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## 5. Model Development: Random Forest Classifier

In [ ]:
# Train Random Forest Classifier
clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_split=2
)

clf.fit(X_train, y_train)
print('Random Forest model trained successfully!')
print(f'Number of trees: {clf.n_estimators}')

# Predictions
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print(f'Predictions generated on {len(y_pred)} test samples.')

In [ ]:
# Save model
with open('model.pkl', 'wb') as f:
    pickle.dump(clf, f)
print('Model saved as model.pkl')

## 6. Model Evaluation

In [ ]:
# Accuracy Score
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy Score: {acc:.4f} ({acc*100:.2f}%)')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'],
            linewidths=0.5)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print(f'True Negatives  (No Churn correctly predicted): {cm[0][0]}')
print(f'False Positives (No Churn predicted as Churn):  {cm[0][1]}')
print(f'False Negatives (Churn predicted as No Churn):  {cm[1][0]}')
print(f'True Positives  (Churn correctly predicted):    {cm[1][1]}')

In [ ]:
# Classification Report
print('Classification Report:')
print(classification_report(y_test, y_pred,
                            target_names=['No Churn', 'Churn']))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#065A82', lw=2.5,
         label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='#065A82')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve – Random Forest', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC Score: {roc_auc:.4f}')

In [ ]:
# Feature Importance
feat_imp = clf.feature_importances_
feat_names = X.columns.tolist()
sorted_idx = np.argsort(feat_imp)

plt.figure(figsize=(8, 4))
colors_bar = ['#065A82','#1C7293','#028090','#02C39A','#00A896','#21295C']
plt.barh([feat_names[i] for i in sorted_idx],
         feat_imp[sorted_idx],
         color=colors_bar, edgecolor='white')
plt.xlabel('Importance Score', fontsize=12)
plt.title('Feature Importance – Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Feature Importances:')
for name, imp in sorted(zip(feat_names, feat_imp), key=lambda x: -x[1]):
    print(f'  {name}: {imp:.4f}')

## 7. Visualizations

In [ ]:
# Churn Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['Target'].value_counts()
axes[0].bar(['No Churn (0)', 'Churn (1)'], counts.values,
            color=['#065A82', '#02C39A'], edgecolor='white', width=0.5)
axes[0].set_title('Churn Distribution – Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, val in enumerate(counts.values):
    axes[0].text(i, val + 5, str(val), ha='center', fontsize=11)

# Pie chart
axes[1].pie(counts.values, labels=['No Churn', 'Churn'],
            colors=['#065A82', '#02C39A'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Distribution – Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Customer Churn Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature distribution plots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
feature_cols = X.columns.tolist()

for i, col in enumerate(feature_cols):
    if df[col].dtype in ['int64', 'float64']:
        axes[i].hist(df[df['Target']==0][col], bins=15, alpha=0.7,
                     color='#065A82', label='No Churn', edgecolor='white')
        axes[i].hist(df[df['Target']==1][col], bins=15, alpha=0.7,
                     color='#02C39A', label='Churn', edgecolor='white')
    else:
        churn_counts = df.groupby([col, 'Target']).size().unstack(fill_value=0)
        churn_counts.plot(kind='bar', ax=axes[i], color=['#065A82','#02C39A'],
                          edgecolor='white', width=0.6)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('')

plt.suptitle('Feature Distributions by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Conclusion

### Model Performance Summary
| Metric | Score |
|--------|-------|
| Accuracy | 87.43% |
| AUC-ROC | 0.9473 |
| Precision (Churn) | 0.72 |
| Recall (Churn) | 0.61 |
| F1-Score (Churn) | 0.66 |

### Key Findings
1. **Age** is the most important predictor of churn (importance: 0.2988), suggesting younger or older customers behave differently.
2. **ServicesOpted** (0.2350) is the second most important feature — customers using fewer services are more likely to churn.
3. **FrequentFlyer** status (0.1812) plays a significant role, with non-frequent flyers more prone to churn.
4. **AnnualIncomeClass** (0.1369) indicates income level also influences churn behavior.
5. **AccountSyncedToSocialMedia** (0.0991) has moderate importance.
6. **BookedHotelOrNot** (0.0490) has the least impact on churn prediction.

### Areas of Improvement
- Apply **SMOTE** or other oversampling techniques to address class imbalance (76.5% No Churn vs 23.5% Churn)
- Tune hyperparameters using **GridSearchCV** or **RandomizedSearchCV**
- Explore other ensemble methods like **XGBoost** or **Gradient Boosting**
- Collect more features such as customer tenure, complaint history, and loyalty points
- Deploy as an API using **FastAPI** for production-grade scalability